In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import json

# Load the dataset
data_path = "dataset.csv"
data = pd.read_csv(data_path)

# Drop timestamp columns
data = data.loc[:, ~data.columns.str.contains("time_stamp", case=False)]

# Extract label column before transformation
labels = data["label"]

# Identify flex, acc, and gyro columns
flex_cols = [col for col in data.columns if col.startswith("flex_finger")]
acc_x_cols = [col for col in data.columns if col.startswith("acc") and col.endswith("_x")]
acc_y_cols = [col for col in data.columns if col.startswith("acc") and col.endswith("_y")]
acc_z_cols = [col for col in data.columns if col.startswith("acc") and col.endswith("_z")]
gyro_x_cols = [col for col in data.columns if col.startswith("gyro") and col.endswith("_x")]
gyro_y_cols = [col for col in data.columns if col.startswith("gyro") and col.endswith("_y")]
gyro_z_cols = [col for col in data.columns if col.startswith("gyro") and col.endswith("_z")]

# Create summarized columns
data["flex_sum"] = data[flex_cols].sum(axis=1)
data["acc_x_sum"] = data[acc_x_cols].sum(axis=1)
data["acc_y_sum"] = data[acc_y_cols].sum(axis=1)
data["acc_z_sum"] = data[acc_z_cols].sum(axis=1)
data["gyro_x_sum"] = data[gyro_x_cols].sum(axis=1)
data["gyro_y_sum"] = data[gyro_y_cols].sum(axis=1)
data["gyro_z_sum"] = data[gyro_z_cols].sum(axis=1)

# Keep only the summarized features and label
data = data[[
    "flex_sum",
    "acc_x_sum", "acc_y_sum", "acc_z_sum",
    "gyro_x_sum", "gyro_y_sum", "gyro_z_sum",
    "label"
]]


In [10]:

# Shuffle the dataset
data = data.sample(frac=1, random_state=42).reset_index(drop=True)

# Extract features and labels
X = data.drop(columns=["label"])
y, label_mapping = pd.factorize(data["label"])

# Save label mapping
label_dict = {i: label for i, label in enumerate(label_mapping)}
with open("class_labels.json", "w") as f:
    json.dump(label_dict, f, indent=4)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)


In [11]:

# Define parameters for LightGBM
params = {
    "objective": "multiclass",
    "num_class": len(pd.unique(y)),  # Number of classes
    "boosting_type": "gbdt",
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 5,
    "verbosity": -1,
    "random_state": 42,
     "min_data_in_leaf": 20
}

# Train the model
model = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=1200)

# Make predictions
y_pred = model.predict(X_test)
y_pred = [list(pred).index(max(pred)) for pred in y_pred]  # Convert probabilities to class indices

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Save the model for deployment
model.save_model("model_lightgbm.txt")
print("Model saved successfully!")

Test Accuracy: 0.9904
Model saved successfully!


In [13]:
import numpy as np

In [52]:
test = np.array([[150, -231.978987, 1.733402, 5.135563	, 19.282443, -209.503817, 86.625954]])

In [15]:
test

array([[  50.      , -164.340396,  124.622985, -103.032074,  203.328244,
         141.068702,  147.068702]])

In [23]:
X_test[1:3]

,flex_sum,acc_x_sum,acc_y_sum,acc_z_sum,gyro_x_sum,gyro_y_sum,gyro_z_sum
93,50,-164.340396,124.622985,-103.032074,203.328244,141.068702,147.068702
6,28,-167.867055,142.715967,-85.456240,-30.603053,-71.442748,-34.572519


In [21]:
y_test[1]

array([1, 1])

In [53]:
model.predict(test)

array([[3.51272043e-08, 1.05637173e-07, 1.50229007e-06, 2.11374487e-06,
        9.99996243e-01]])

In [49]:
X_test[0:1]

,flex_sum,acc_x_sum,acc_y_sum,acc_z_sum,gyro_x_sum,gyro_y_sum,gyro_z_sum
275,26,-231.978987,1.733402,5.135563,19.282443,-209.503817,86.625954


In [32]:
data.head(20)

,flex_sum,acc_x_sum,acc_y_sum,acc_z_sum,gyro_x_sum,gyro_y_sum,gyro_z_sum,label
0,50,-169.389767,43.689392,34.249055,12.977099,-93.480916,-566.053435,alsalam_alikum
1,36,-194.739575,-5.913678,49.413928,-253.053435,-2.977099,180.274809,alsalam_alikum
2,33,-165.523132,107.803718,-79.928028,-196.641221,-38.389313,-110.480916,good_morning
3,37,-188.885752,44.721293,20.305224,-84.519084,-144.564885,35.610687,alsalam_alikum
4,125,-130.773689,-108.708726,163.895075,113.519084,109.755725,745.610687,alsalam_alikum
5,23,-169.531025,118.228072,-75.395804,-9.778626,-15.297710,21.099237,good_morning
6,28,-167.867055,142.715967,-85.456240,-30.603053,-71.442748,-34.572519,good_morning
7,40,-199.559103,-16.292542,-4.194641,-117.488550,181.916031,283.526718,alsalam_alikum
8,20,-224.439646,38.996757,23.994689,-132.358779,-105.916031,25.183206,wht_ur_name
9,37,-176.301828,127.476873,-75.019915,-5.656489,-143.183206,-38.465649,good_morning


In [33]:
# Load the dataset
data_path = "dataset.csv"
data = pd.read_csv(data_path)

In [37]:
count = [i for i in data.columns if "time_stamp" in i]

In [39]:
len(count)

25